# Coleta de notícias sobre Juizados Especiais


Usei requisições HTTP e Beautiful Soup para coletar as dez notícias mais recentes da página dos Juizados Especiais no portal do CNJ. Como o conteúdo está disponível no HTML da página, não foi necessário utilizar Selenium.

In [2]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

#feito com auxilio de IA chat GPT

PASTA_PROJETO = Path.cwd()
if PASTA_PROJETO.name == "src":
    PASTA_PROJETO = PASTA_PROJETO.parent
ARQUIVO_BRUTO = PASTA_PROJETO / "data" / "raw" / "noticias_juizados.html"
ARQUIVO_TRATADO = PASTA_PROJETO / "data" / "processed" / "noticias_juizados.csv"

In [3]:
resposta = requests.get(
    "https://www.cnj.jus.br/programas-e-acoes/juizados-especiais/noticias/",
    headers={"User-Agent": "Mozilla/5.0 (projeto acadêmico TP3)"},
    timeout=120,
)
print("Status da página:", resposta.status_code)
resposta.raise_for_status()
ARQUIVO_BRUTO.write_text(resposta.text, encoding="utf-8")

Status da página: 200


283591

In [4]:
artigos = BeautifulSoup(resposta.text, "html.parser").select("article.elementor-post")[:10]
print("Notícias encontradas:", len(artigos))
if not artigos:
    raise RuntimeError("Nenhuma notícia encontrada. Verifique o seletor da página do CNJ.")
print(artigos[0].select_one(".elementor-post__title a").get_text(" ", strip=True))

Notícias encontradas: 10
Juizados especiais de Teresina se destacam em produtividade e celeridade processual


In [5]:
linhas = []

for artigo in artigos:
    titulo = artigo.select_one(".elementor-post__title a")
    linhas.append({
        "titulo": titulo.get_text(" ", strip=True),
        "data": artigo.select_one(".elementor-post-date").get_text(" ", strip=True),
        "resumo": artigo.select_one(".elementor-post__excerpt").get_text(" ", strip=True),
        "url": titulo["href"],
        "fonte": "Conselho Nacional de Justiça",
    })

noticias = pd.DataFrame(linhas)
noticias

,titulo,data,resumo,url,fonte
0,Juizados especiais de Teresina se destacam em ...,9 de outubro de 2026,Os Juizados Especiais Cíveis e Criminais (JECC...,https://www.cnj.jus.br/juizados-especiais-de-t...,Conselho Nacional de Justiça
1,Boas práticas e parceria institucional fortale...,21 de setembro de 2026,O fortalecimento dos Juizados Especiais passa ...,https://www.cnj.jus.br/boas-praticas-e-parceri...,Conselho Nacional de Justiça
2,Fórum dos Juizados Especiais será realizado em...,1 de setembro de 2026,"A Paraíba vai receber, entre os dias 17 e 19 d...",https://www.cnj.jus.br/forum-dos-juizados-espe...,Conselho Nacional de Justiça
3,Juizado na Comunidade leva serviços de cidadan...,30 de junho de 2026,A Central dos Juizados Especiais do Tribunal d...,https://www.cnj.jus.br/juizado-na-comunidade-l...,Conselho Nacional de Justiça
4,Atuação dos Juizados Especiais aproxima a Just...,16 de junho de 2026,Com o tema “Fortalecer os Juizados Especiais é...,https://www.cnj.jus.br/atuacao-dos-juizados-es...,Conselho Nacional de Justiça
5,Tribunais promovem ações e mutirões processuai...,15 de junho de 2026,Com o tema “Fortalecer os Juizados Especiais é...,https://www.cnj.jus.br/tribunais-promovem-acoe...,Conselho Nacional de Justiça
6,CNJ reforça atuação em rede e inovação durante...,29 de maio de 2026,O Conselho Nacional de Justiça (CNJ) participo...,https://www.cnj.jus.br/cnj-reforca-atuacao-em-...,Conselho Nacional de Justiça
7,Juizado Especial do Torcedor e Grupo de Trabal...,7 de maio de 2026,O Juizado Especial de Defesa do Torcedor de Sã...,https://www.cnj.jus.br/juizado-especial-do-tor...,Conselho Nacional de Justiça
8,Unidade Móvel de Trânsito do TJRN fecha 2025 c...,27 de janeiro de 2026,A Unidade Móvel de Trânsito (UMT) do Tribunal ...,https://www.cnj.jus.br/unidade-movel-de-transi...,Conselho Nacional de Justiça
9,TJPB adota política para equilibrar distribuiç...,20 de janeiro de 2026,"Na última semana, o Órgão Especial do Tribunal...",https://www.cnj.jus.br/tjpb-adota-politica-par...,Conselho Nacional de Justiça


In [6]:
assert len(noticias) == 10
assert noticias["titulo"].notna().all()
assert noticias["url"].nunique() == 10

print("Testes concluídos sem erros.")

Testes concluídos sem erros.


In [7]:
noticias.to_csv(ARQUIVO_TRATADO, index=False, encoding="utf-8-sig")
print("Arquivo bruto:", ARQUIVO_BRUTO)
print("Arquivo tratado:", ARQUIVO_TRATADO)

Arquivo bruto: c:\Carol\Facul\2 ano - 2 bi\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP3\PB_Juizados_Especiais\data\raw\noticias_juizados.html
Arquivo tratado: c:\Carol\Facul\2 ano - 2 bi\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP3\PB_Juizados_Especiais\data\processed\noticias_juizados.csv


## Preparação para uso futuro com LLMs

Os títulos e resumos foram organizados em um único campo textual, mantendo data, URL e fonte para rastreabilidade. O arquivo poderá ser utilizado futuramente em tarefas como geração de resumos, análise de sentimentos e perguntas e respostas. 

In [8]:
ARQUIVO_LLM = (
    PASTA_PROJETO
    / "data"
    / "processed"
    / "noticias_para_llm.csv"
)

noticias_llm = noticias[
    ["titulo", "data", "resumo", "url", "fonte"]
].copy()

noticias_llm["titulo"] = (
    noticias_llm["titulo"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

noticias_llm["resumo"] = (
    noticias_llm["resumo"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

noticias_llm.insert(
    0,
    "id_texto",
    [
        f"noticia_{indice:03d}"
        for indice in range(1, len(noticias_llm) + 1)
    ],
)

noticias_llm["texto"] = (
    "Título: "
    + noticias_llm["titulo"]
    + "\nResumo: "
    + noticias_llm["resumo"]
)

noticias_llm = noticias_llm[
    [
        "id_texto",
        "titulo",
        "resumo",
        "texto",
        "data",
        "url",
        "fonte",
    ]
]

assert noticias_llm["texto"].notna().all()
assert noticias_llm["url"].nunique() == len(noticias_llm)

noticias_llm.to_csv(
    ARQUIVO_LLM,
    index=False,
    encoding="utf-8-sig",
)

print("Textos preparados:", len(noticias_llm))
print("Arquivo criado:", ARQUIVO_LLM)

noticias_llm.head()

Textos preparados: 10
Arquivo criado: c:\Carol\Facul\2 ano - 2 bi\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP3\PB_Juizados_Especiais\data\processed\noticias_para_llm.csv


,id_texto,titulo,resumo,texto,data,url,fonte
0,noticia_001,Juizados especiais de Teresina se destacam em ...,Os Juizados Especiais Cíveis e Criminais (JECC...,Título: Juizados especiais de Teresina se dest...,9 de outubro de 2026,https://www.cnj.jus.br/juizados-especiais-de-t...,Conselho Nacional de Justiça
1,noticia_002,Boas práticas e parceria institucional fortale...,O fortalecimento dos Juizados Especiais passa ...,Título: Boas práticas e parceria institucional...,21 de setembro de 2026,https://www.cnj.jus.br/boas-praticas-e-parceri...,Conselho Nacional de Justiça
2,noticia_003,Fórum dos Juizados Especiais será realizado em...,"A Paraíba vai receber, entre os dias 17 e 19 d...",Título: Fórum dos Juizados Especiais será real...,1 de setembro de 2026,https://www.cnj.jus.br/forum-dos-juizados-espe...,Conselho Nacional de Justiça
3,noticia_004,Juizado na Comunidade leva serviços de cidadan...,A Central dos Juizados Especiais do Tribunal d...,Título: Juizado na Comunidade leva serviços de...,30 de junho de 2026,https://www.cnj.jus.br/juizado-na-comunidade-l...,Conselho Nacional de Justiça
4,noticia_005,Atuação dos Juizados Especiais aproxima a Just...,Com o tema “Fortalecer os Juizados Especiais é...,Título: Atuação dos Juizados Especiais aproxim...,16 de junho de 2026,https://www.cnj.jus.br/atuacao-dos-juizados-es...,Conselho Nacional de Justiça
